# Medical NER Project - Notebook 02
## Preprocessing: Converting MACCROBAT Annotations to BIO Format


This notebook performs data preprocessing by converting the MACCROBAT2018 annotated dataset into the BIO (Beginning, Inside, Outside) tagging scheme, which is commonly used for sequence labeling tasks such as Named Entity Recognition (NER).

The process includes:
- Tokenization and sentence segmentation with start-end character indexing
- Punctuation stripping and stopword filtering
- Alignment of annotation spans with token positions
- Conversion of annotations to the BIO format
- Saving processed data to `.bio` files

All processing steps are designed to be compatible with training biomedical transformers like BioBERT.


In [1]:
import os
import re
import json
import shutil

In [2]:
import string

import nltk
from nltk.corpus import stopwords

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to /Users/karan/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [3]:
!pwd

/Users/karan/Documents/Projects/biobert-medical-ner/notebooks


In [4]:

data_dir = "../data/annotated_json_data"

### Entity Label Mapping

Entity labels are mapped to short acronyms to simplify BIO tag representation and downstream processing. A bidirectional dictionary is constructed to facilitate quick lookups during annotation conversion.


In [5]:
entity_to_acronyms = {
    'Activity': 'ACT',
    'Administration': 'ADM',
    'Age': 'AGE',
    'Area': 'ARA',
    'Biological_attribute': 'BAT',
    'Biological_structure': 'BST',
    'Clinical_event': 'CLE',
    'Color': 'COL',
    'Coreference': 'COR',
    'Date': 'DAT',
    'Detailed_description': 'DET',
    'Diagnostic_procedure': 'DIA',
    'Disease_disorder': 'DIS',
    'Distance': 'DIS',
    'Dosage': 'DOS',
    'Duration': 'DUR',
    'Family_history': 'FAM',
    'Frequency': 'FRE',
    'Height': 'HEI',
    'History': 'HIS',
    'Lab_value': 'LAB',
    'Mass': 'MAS',
    'Medication': 'MED',
    'Nonbiological_location': 'NBL',
    'Occupation': 'OCC',
    'Other_entity': 'OTH',
    'Other_event': 'OTE',
    'Outcome': 'OUT',
    'Personal_background': 'PER',
    'Qualitative_concept': 'QUC',
    'Quantitative_concept': 'QUC',
    'Severity': 'SEV',
    'Sex': 'SEX',
    'Shape': 'SHA',
    'Sign_symptom': 'SIG',
    'Subject': 'SUB',
    'Texture': 'TEX',
    'Therapeutic_procedure': 'THP',
    'Time': 'TIM',
    'Volume': 'VOL',
    'Weight': 'WEI'
}


acronyms_to_entities = {v: k for k, v in entity_to_acronyms.items()}

### Load Annotated JSON Data

The dataset is stored as a JSON file with raw text and associated span-based annotations. This section loads the data into memory for preprocessing.


In [6]:
# Open the JSON file for reading
with open(os.path.join(data_dir, "annotated_data.json"), 'r') as f:

    # Load the JSON data into a dictionary
    data = json.load(f)

### Text Cleaning and Tokenization

The `split_text` function performs rule-based tokenization using regular expressions and indexes the start and end positions of each token in the original text. This granularity ensures precise alignment between entity annotations and tokens.

- `remove_trailing_punctuation(token)`: Removes non-word trailing punctuation from tokens.
- `split_text(text)`: Splits the document into token sequences, sentence breaks, and positional index ranges.


In [7]:
def remove_trailing_punctuation(token):
    """
    Removes trailing punctuation from a token.

    Args:
        token (str): A string representing the token to be cleaned.

    Returns:
        str: The cleaned token with trailing punctuation removed.
    """
    while token and re.search(r'[^\w\s\']', token[-1]):
        token = token[:-1]
        
    return token

`split_text` function that takes in a text as input and returns three lists:

* tokens: a list of words (with trailing punctuation removed)
* start_end_ranges: a list of tuples representing the start and end indices of each word in the original text
* sentence_breaks: a list of indices indicating the positions in the tokens list where a new sentence begins.

The function first defines a regular expression pattern to match non-space and non-dash characters. It then initializes empty lists for tokens, start_end_ranges, and sentence_breaks.


The function then iterates over each sentence in the input text, finds the words in each sentence using regex matching, removes trailing punctuation from each word using another function, and calculates the start and end indices for each word.


The function updates the start and end indices to account for the sentence's position in the entire text, adds the indices and words to the respective lists, and appends the index of the last word in the sentence to the sentence_breaks list.


Finally, the function returns the three lists containing the processed data.

In [8]:
def split_text(text):

    regex_match = r'[^\s\u200a\-\u2010-\u2015\u2212\uff0d]+'  # r'[^\s\u200a\-\—\–]+'

    tokens = []
    start_end_ranges = []

    sentence_breaks = []

    start_idx = 0

    for sentence in text.split('\n'):
        words = [match.group(0) for match in re.finditer(regex_match, sentence)]
        processed_words = list(map(remove_trailing_punctuation, words))
        sentence_indices = [(match.start(), match.start() + len(token)) for match, token in
                            zip(re.finditer(regex_match, sentence), processed_words)]

        # Update the indices to account for the current sentence's position in the entire text
        sentence_indices = [(start_idx + start, start_idx + end) for start, end in sentence_indices]

        start_end_ranges.extend(sentence_indices)
        tokens.extend(processed_words)

        sentence_breaks.append(len(tokens))

        start_idx += len(sentence) + 1
    return tokens, start_end_ranges, sentence_breaks

In [9]:
for doc_id, doc in data.items():
    print(split_text(doc['text'][:100]))
    break

(['Our', '24', 'year', 'old', 'non', 'smoking', 'male', 'patient', 'presented', 'with', 'repeated', 'hemoptysis', 'in', 'May', '2008', 'with', '4', 'days'], [(0, 3), (4, 6), (7, 11), (12, 15), (16, 19), (20, 27), (28, 32), (33, 40), (41, 50), (51, 55), (56, 64), (65, 75), (76, 78), (79, 82), (83, 87), (88, 92), (93, 94), (95, 99)], [18])


### BIO Tagging Strategy

The annotations are mapped to the corresponding tokens using character offset alignment. Each matched token is tagged in the BIO format based on its position within an annotated span.

- `tag_token(...)`: Assigns a `B-` or `I-` tag depending on the context and continuity of the same entity.
- `write_bio_files(...)`: Outputs token-tag sequences into `.bio` formatted files.
- `convert_ann_to_bio(...)`: Converts all annotated documents into BIO sequences and optionally filters for specific entity types.


In [10]:
def tag_token(tokens, tags, token_pos, entity):
    """
    Modifies a list of tags by adding a tag label to a token at a given position in the list, based on the position of the 
    previous token and whether the current token has the same tag label as the previous token.

    Args:
    - tokens (list): A list of tokens in a sequence.
    - tags (list): A list of tag labels corresponding to the tokens in a sequence.
    - token_pos (int): The position of the token to tag.
    - entity (str): The tag label to add to the token.

    Returns:
    - tags (list): The modified list of tag labels.
    """
    
    stop_words = stopwords.words('english')
    
    tag = entity_to_acronyms[entity]
    
    if token_pos > 0 and f'{tag}' in tags[token_pos - 1]:        
            tags[token_pos] = f'I-{tag}'
    elif tokens[token_pos] not in stop_words:
            tags[token_pos] = f'B-{tag}'
            
    return tags


In [11]:
def write_bio_files(output_file_path, tokens, tags, sentence_breaks):

    # Write the tags to a .bio file
    with open(output_file_path, 'w') as f:
        for i in range(len(tokens)):
            token = tokens[i].strip()
            if token:
                if i in sentence_breaks:
                    f.write("\n")
                f.write(f"{tokens[i]}\t{tags[i]}\n")


In [12]:
def convert_ann_to_bio(data, output_dir, filtered_entities=[]):
    
    """
    Convert annotations from a dictionary of text files to a BIO-tagged sequence.

    Args:
        data (dict): A dictionary of text files where keys are file IDs and values are dictionaries containing 'text' and
            'annotations' keys.
        filtered_entities (list): A list of entity labels to include. If provided, only annotations with these labels will
            be converted to the BIO format. Defaults to an empty list.

    Returns:
        A tuple of two lists: tokens and tags.
        - tokens (list): A list of tokens in a sequence.
        - tags (list): A list of corresponding tags for each token in the sequence. Tags are BIO formatted.

    """
    
    if os.path.exists(output_dir):
        # Delete the contents of the directory
        shutil.rmtree(output_dir)
    # Recreate the directory
    os.makedirs(output_dir)
    
    
    for file_id in data:
        text = data[file_id]['text']
        annotations = data[file_id]['annotations']
        
        # Tokenizing
        tokens, token2text, sentence_breaks = split_text(text)

        # Initialize the tags
        tags = ['O'] * len(tokens)

        ann_pos = 0
        token_pos = 0

        while ann_pos < len(annotations) and token_pos < len(tokens):

            label = annotations[ann_pos]['label']
            start = annotations[ann_pos]['start']
            end = annotations[ann_pos]['end']

            if filtered_entities:
                if label not in filtered_entities:
                    # increment to access next annotation
                    ann_pos += 1
                    continue
            
            ann_word = text[start:end]

            # find the next word that fall between the annotation start and end
            while token_pos < len(tokens) and token2text[token_pos][0] < start:
                
                token_pos += 1

            if tokens[token_pos] == ann_word or \
                ann_word in tokens[token_pos] or \
                re.sub(r'\W+', '', ann_word) in re.sub(r'\W+', '', tokens[token_pos]):
                tag_token(tokens, tags, token_pos, label)
            elif ann_word in tokens[token_pos - 1] or \
                ann_word in tokens[token_pos - 1] or \
                re.sub(r'\W+', '', ann_word) in re.sub(r'\W+', '', tokens[token_pos - 1]):
                tag_token(tokens, tags, token_pos - 1, label)
            else:
                print(tokens[token_pos], tokens[token_pos - 1], ann_word, label)

            # increment to access next annotation
            ann_pos += 1

        # write to bio file
        write_bio_files(os.path.join(output_dir, f"{file_id}.bio"), tokens, tags, sentence_breaks)
    print("Conversion complete")

In [13]:
convert_ann_to_bio(data, '../data/bio_data_files')

Conversion complete


### Output Summary

All converted `.bio` files are saved in the `../data/bio_data_files/` directory. Each file corresponds to a document and contains token-wise BIO-labeled data suitable for NER model training.

If required, users can pass a `filtered_entities` list to focus the model on a subset of relevant biomedical entity classes.


This preprocessing pipeline ensures robust and reproducible alignment between character-level annotations and token-level representations, enabling high-quality supervised training of transformer-based NER models in the biomedical domain.

